Crude Oil Forecasting data cleaning and preprocessing.

In [1]:
import pandas as pd
import numpy as np

In [17]:
df = pd.read_excel("crude_oil.xlsx")
df.head()
# RWTC signifies the crude oil price benchmark in the United States
# RBRTE signifies the Europe and global crude oil price benchmark
# The price is in USD per barrel
# This dataset shows the monthly crude oil price from 1986 till July 2026

,Sourcekey,RWTC,RBRTE
0,Date,"Cushing, OK WTI Spot Price FOB (Dollars per Ba...",Europe Brent Spot Price FOB (Dollars per Barrel)
1,1986-01-15 00:00:00,22.93,NaN
2,1986-02-15 00:00:00,15.46,NaN
3,1986-03-15 00:00:00,12.61,NaN
4,1986-04-15 00:00:00,12.84,NaN


In [24]:
# check missing value
df.isna().sum()
# RBRTE has 16 missing values

# RBRTE contains 16 missing values because Brent price data begins later than WTI price data.
# These values are kept as NaN to preserve the full WTI time series.

,0
Sourcekey,0
RWTC,0
RBRTE,16


In [25]:
# check duplicate
df[df.duplicated()]
# no duplicated value

,Sourcekey,RWTC,RBRTE


In [26]:
# removing descriptive row
df = df.drop(index=0)

# reset the index
df = df.reset_index(drop=True)

df.head()

,Sourcekey,RWTC,RBRTE
0,1986-01-15 00:00:00,22.93,NaN
1,1986-02-15 00:00:00,15.46,NaN
2,1986-03-15 00:00:00,12.61,NaN
3,1986-04-15 00:00:00,12.84,NaN
4,1986-05-15 00:00:00,15.38,NaN


In [28]:
# renaming columns
df = df.rename(columns={
    "Sourcekey" : "Date",
    "RWTC" : "WTI_price",
    "RBRTE" : "Brent_price"
})

df.head()

,Date,WTI_price,Brent_price
0,1986-01-15 00:00:00,22.93,NaN
1,1986-02-15 00:00:00,15.46,NaN
2,1986-03-15 00:00:00,12.61,NaN
3,1986-04-15 00:00:00,12.84,NaN
4,1986-05-15 00:00:00,15.38,NaN


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487 entries, 0 to 486
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Date         487 non-null    object
 1   WTI_price    487 non-null    object
 2   Brent_price  471 non-null    object
dtypes: object(3)
memory usage: 11.5+ KB


In [30]:
df["Date"] = pd.to_datetime(df["Date"])
df["WTI_price"] = df["WTI_price"].astype(float)
df["Brent_price"] = df["Brent_price"].astype(float)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487 entries, 0 to 486
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Date         487 non-null    datetime64[ns]
 1   WTI_price    487 non-null    float64       
 2   Brent_price  471 non-null    float64       
dtypes: datetime64[ns](1), float64(2)
memory usage: 11.5 KB


In [31]:
# check if the dates are sorted chronologically
df["Date"].is_monotonic_increasing

True

In [32]:
print("Start date:", df["Date"].min())
print("End date:", df["Date"].max())
print("Number of observations:", len(df))

Start date: 1986-01-15 00:00:00
End date: 2026-07-15 00:00:00
Number of observations: 487


In [33]:
# Create the expected monthly date sequence
expected_dates = pd.date_range(
    start=df["Date"].min(),
    end=df["Date"].max(),
    freq="MS"
) + pd.Timedelta(days=14)

In [34]:
# Find missing months
missing_dates = expected_dates.difference(df["Date"])

print("Number of missing months:", len(missing_dates))
print(missing_dates)

Number of missing months: 0
DatetimeIndex([], dtype='datetime64[ns]', freq=None)


In [35]:
df.describe()

,Date,WTI_price,Brent_price
count,487,487.000000,471.000000
mean,2006-04-15 11:40:46.817248512,48.599466,51.416136
min,1986-01-15 00:00:00,11.350000,9.820000
25%,1996-02-29 12:00:00,20.280000,19.480000
50%,2006-04-15 00:00:00,44.650000,46.520000
75%,2016-05-30 12:00:00,71.615000,74.800000
max,2026-07-15 00:00:00,133.880000,132.720000
std,NaN,29.487472,32.806558


In [36]:
# Final validation of the cleaned dataset
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:")
print(df.dtypes)

Dataset shape: (487, 3)

Missing values:
Date            0
WTI_price       0
Brent_price    16
dtype: int64

Duplicate rows: 0

Data types:
Date           datetime64[ns]
WTI_price             float64
Brent_price           float64
dtype: object


In [37]:
df.to_csv("crude_oil_clean.csv", index = False)